# LeafCheck – Step 1: train the model (Google Colab)

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
It takes about 15–25 minutes. Everything is saved to Google Drive in `MyDrive/leafcheck/`,
so a disconnect does not lose the model.

What this notebook does:
1. Loads the public **PlantVillage** colour dataset.
2. Keeps only our **19 classes** (tomato, potato, corn, bell pepper) and **at most 300 images per class**.
3. Makes a **stratified 70 / 15 / 15** train / validation / test split with seed 42.
4. Trains **MobileNetV2** with transfer learning (ImageNet weights, base frozen, new 19-class head).
5. Saves `model/leafcheck_mobilenetv2.keras`, `model/labels.json`, the test split and training curves.

## 0. Setup: check GPU, fix random seeds, mount Google Drive

In [ ]:
import json, random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IMG_SIZE = 224
MAX_PER_CLASS = 300   # cap from the brief; drop to 150 if short on time
BATCH_SIZE = 32
EPOCHS = 10

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - switch runtime to GPU!")

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/leafcheck")  # mirrors the repo layout
for sub in ("model", "reports", "samples", "test_split"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
print("Saving outputs to", OUT_DIR)

In [ ]:
CLASS_NAMES = [
    "Corn___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn___Common_rust",
    "Corn___Northern_Leaf_Blight",
    "Corn___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]  # same list and order as leafcheck/config.py

## 1. Load PlantVillage and keep our 19 classes

We read the images as **encoded JPEG bytes** (no decoding yet), which is fast and uses little memory.
Source A is TensorFlow Datasets (`plant_village`). If its download server is down, source B
clones only the needed folders from the original PlantVillage GitHub repository.

Folder names differ slightly between sources (`Corn_(maize)` vs `Corn`, trailing `_`),
so `normalise()` maps both to the names in `CLASS_NAMES`.

In [ ]:
def normalise(name: str) -> str:
    """Map a dataset class name to our CLASS_NAMES spelling."""
    return name.replace("Corn_(maize)", "Corn").rstrip("_").strip()

WANTED = set(CLASS_NAMES)
images_by_class = {name: [] for name in CLASS_NAMES}  # class name -> list of JPEG bytes

def load_from_tfds():
    import tensorflow_datasets as tfds
    ds, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True,
                         decoders={"image": tfds.decode.SkipDecoding()})
    names = info.features["label"].names
    for image_bytes, label in ds.as_numpy_iterator():
        name = normalise(names[label])
        if name in WANTED:
            images_by_class[name].append(image_bytes)

def load_from_github():
    import subprocess
    repo = Path("/content/PlantVillage-Dataset")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/spMohanty/PlantVillage-Dataset", str(repo)], check=True)
    color_dir = repo / "raw" / "color"
    folders = [f"raw/color/{n.replace('Corn___', 'Corn_(maize)___')}" for n in CLASS_NAMES]
    folders += [f + "_" for f in folders]  # some folders end with "_"
    subprocess.run(["git", "-C", str(repo), "sparse-checkout", "set", *folders], check=True)
    for folder in color_dir.iterdir():
        name = normalise(folder.name)
        if name in WANTED:
            images_by_class[name] = [p.read_bytes() for p in sorted(folder.glob("*")) if p.suffix.lower() in (".jpg", ".jpeg", ".png")]

try:
    load_from_tfds()
    print("Loaded from TensorFlow Datasets")
except Exception as error:
    print("TFDS failed:", error, "\nFalling back to GitHub source...")
    images_by_class = {name: [] for name in CLASS_NAMES}
    load_from_github()

missing = [n for n, imgs in images_by_class.items() if not imgs]
assert not missing, f"Classes not found: {missing}"
for name, imgs in images_by_class.items():
    print(f"{len(imgs):5d}  {name}")

## 2. Cap at 300 images per class and make a stratified 70 / 15 / 15 split

**Why cap?** Faster training and less class imbalance (some classes have 5,000+ images, others ~150).

**Why stratified?** Every split gets the same class proportions, so the test set is not missing
a rare class by bad luck. **Why a fixed seed?** The split is reproducible.

The **test set is never used for training or for choosing epochs** – only for the final report.

In [ ]:
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(SEED)
all_bytes, all_labels = [], []
for class_id, name in enumerate(CLASS_NAMES):
    imgs = images_by_class[name]
    chosen = rng.choice(len(imgs), size=min(MAX_PER_CLASS, len(imgs)), replace=False)
    all_bytes += [imgs[i] for i in chosen]
    all_labels += [class_id] * len(chosen)
del images_by_class  # free memory

all_labels = np.array(all_labels)
x_train, x_temp, y_train, y_temp = train_test_split(
    all_bytes, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

print(f"total {len(all_bytes)} | train {len(x_train)} | val {len(x_val)} | test {len(x_test)}")
print("per-class counts in test:", np.bincount(y_test, minlength=len(CLASS_NAMES)))

## 3. Save the test split to Drive right away

`evaluate.ipynb` and the repo's `tests/test_model_sanity.py` use exactly these held-out images.

In [ ]:
test_dir = OUT_DIR / "test_split"
test_index = {}
for i, (image_bytes, label) in enumerate(zip(x_test, y_test)):
    filename = f"test_{i:04d}.jpg"
    (test_dir / filename).write_bytes(image_bytes)
    test_index[filename] = CLASS_NAMES[label]
(test_dir / "expected.json").write_text(json.dumps(test_index, indent=1))
print("saved", len(test_index), "test images")

## 4. Input pipeline: decode → resize 224×224 → augment (train only) → normalise

* `preprocess_input` scales pixels from [0, 255] to [-1, 1], the range MobileNetV2 was trained on.
  The web app does the same with `x / 127.5 - 1` in `leafcheck/preprocess.py`.
* **Augmentation** (random flip, small rotation, small brightness change) only on training data:
  it shows the model slightly different versions of each leaf so it memorises less (less overfitting).

In [ ]:
NUM_CLASSES = len(CLASS_NAMES)
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),               # about +/- 18 degrees
    tf.keras.layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="augment")

def decode(image_bytes, label):
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), antialias=True)
    return image, tf.one_hot(label, NUM_CLASSES)

def make_dataset(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda img, lab: (augment(img, training=True), lab), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda img, lab: (preprocess_input(img), lab), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val, training=False)

## 5. Build the model: MobileNetV2 + small new head

**Transfer learning:** MobileNetV2 already learned general visual features (edges, textures,
colours) from 1.4 million ImageNet photos. We **freeze** those layers and train only a small
new head for our 19 classes. This needs far less data and time than training from scratch.

**Why MobileNetV2?** Small (~3.5 M parameters, ~14 MB file) and fast on a laptop CPU
(~300 M multiply-adds per image), while still accurate on PlantVillage.

Head: `GlobalAveragePooling2D` (7×7×1280 → 1280 numbers) → `Dropout(0.2)` → `Dense(19, softmax)`.

In [ ]:
base = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs, name="leafcheck_mobilenetv2")

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary(show_trainable=True)

## 6. Train with early stopping

**EarlyStopping** watches validation loss; if it does not improve for 3 epochs, training stops
and the best weights are restored. This is our main guard against **overfitting**
(train accuracy going up while validation accuracy stops improving).

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True)

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[early_stop])

## 7. Save model + labels to Drive immediately (Colab can disconnect)

In [ ]:
model.save(OUT_DIR / "model" / "leafcheck_mobilenetv2.keras")
(OUT_DIR / "model" / "labels.json").write_text(json.dumps(CLASS_NAMES, indent=1))
(OUT_DIR / "reports" / "history.json").write_text(json.dumps(history.history, indent=1))
print("saved model, labels and history to", OUT_DIR)

## 8. Training curves (how we check for overfitting)

If the train and validation curves stay close, the model is not overfitting badly.
A large gap (train ≫ validation) would mean it memorised the training images.

In [ ]:
import matplotlib.pyplot as plt

h = history.history
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ("accuracy", "loss")):
    ax.plot(h[metric], label="train")
    ax.plot(h["val_" + metric], label="validation")
    ax.set_title(metric); ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "reports" / "training_curves.png", dpi=120)
plt.show()
print(f"best val accuracy: {max(h['val_accuracy']):.3f}")

## 9. (Optional stretch) Fine-tune the last ~20 layers

Only if you are ahead of schedule and validation accuracy is below 90 %. Set `FINE_TUNE = True`
and re-run this cell, then re-run section 7 to save. A tiny learning rate (1e-5) adjusts the
pretrained features gently without destroying them.

In [ ]:
FINE_TUNE = False
if FINE_TUNE:
    base.trainable = True
    for layer in base.layers[:-20]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    history_ft = model.fit(train_ds, validation_data=val_ds, epochs=4, callbacks=[early_stop])

**Done.** Now open `notebooks/evaluate.ipynb` in Colab and run all cells.